In [ ]:
'''
list of errors possible:
upon inputing the file & running:
- the input is not a xlsx or a readable file
- the person stopped the program while it was running
- input file too big --Warning
- missing one of the following columns & not recoverable:
    start location	
    end location	
    start time	
    end time	
    activity	
    line	
    energy consumption	
    bus

Upon reading the file:
    -Energy consumption is 5 if idle, less than 0 if charging, greater than 0 otherwise
    -A bus can’t make two or more trips at the same time.
    -in column "activity":
        - only valid activities are: material trip, service trip, idle, charging
        - activity is blank -- warning
    - in columns "start" and "end" location:
        - must only have values of: ehvbst, ehvapt, ehvgar -- Is a station used only once? | warning
        - Start time must be before end time. Times before 3am are considered hours 25 to 28 | Not final
        - Start and end location must not be the same unless charging or idling.
    - in column "line":
        - A service trip must have a line. A non-service trips line is ignored/replaced with next service trip line.
        - bus line is empty, or (at most) 3 digits
        - if 2 buses are on the same line and idle at the same location & time for the same service trip (look 2 conditions above).
    - in column "energy consumption":
        - only floats/int
        - Energy consumption is 5 if idle, less than 0 if charging, greater than 0 otherwise
    - in column start time and end time:
        - If a start time and an end time of a trip fit wholy inside the times of another trip -- Warning

'''
    

In [2]:
import pandas as pd
import numpy as np
import math
import os

In [ ]:
# main function to read file and check for errors |Psuedo code
def read_file(file):
    # check if its the appropriate file type
    ext = os.path.splitext(file)[1].lower()
    if ext != ".csv" or ".xlsx" or ".xls" or ".pdf":
        raise FileNotFoundError("Your file type is invalid, please input an excel file or csv file")
        return False
    else:
        # check if it fulfills the column requirements
        required_columns = ["start location","end location","start time","end time","activity","line","energy consumption","bus"]
        input_columns = file.columns
        missing_columns = set(required_columns) - set(input_columns)
        if missing:
            raise IndexError(f"Unable to proceed, Missing columns: {missing}, please make sure all columns are inserted")
            return False
        else:
            # now check individual columns for the data errors
            start_location = file["start location"]
            end_location = file["end location"]
            start_time = file["start time"]
            end_time = file["end time"]
            activity = file["activity"]
            line = file["line"]
            energy_consumption = file["energy consumption"]
            bus = file["bus"]

            # check if start & end locations are valid
            allowed_locations = ["ehvbst", "ehvapt", "ehvgar"]
            if not all start_location.isin(allowed_locations).all() and end_location.isin(allowed_locations).all():
                # store these invalid locations
                bad_values = file.loc[~file["status"].isin(allowed), "status"].unique()
                raise ValueError(f"Unexpected values found: {bad_values}. Are these locations correct?")
            elif start_location == end_location: # check if the start location and end location is the same
                raise ValueError(f"start and end locations are the same for row # {}, is this correct?")

            # check the column "activity"
            allowed_activities = ["idle","service trip", "material trip","charging"]
            if not all activities.isin(allowed_activities).all():
                raise ValueError(f"The activities in row {row number} are invalid, the activities allowed are {allowed_activities}")
            elif activities.isnull():
                raise ValueError(f"Warning, row number {row number} has an an empty activity")

            # check the column "energy"
            for idx, value in energy.items(): # if any values in the column "energy" is not a number
                is_valid = not pd.isna(pd.to_numeric(value, errors="coerce"))
                if not is_valid:
                    return TypeError(f"Row {idx}: '{value}' is not a number or a decimal")
                elif energy == 5 and activity != "idle":
                    raise ValueError(f"In row # {row number}, idle activites must consume 5 energy") #if the activity is idle, the energy must be equal to 5
                    
            for idx,value in line.items(): # if any values in the column line is not a number. 
                is_valid = not pd.isna(pd.to_numeric(value, errors="coerce"))
                if not is_valid:
                    return TypeError(f"Row {idx}: '{value}' is not a valid line number. Lines must be empty or have numbers with max 3 digits")
                elif len(pd.to_numeric(value, errors="coerce")) > 3:
                    return IndexError(f"Row {idx}: '{value}' is not a valid line number. It has more than 3 digits.")
            # if the exact same bus has the exact same start time then it raises a warning
            elif line == start_time:
                raise ValueError(f"The rows {row1}, {row2}, have the exact same bus line {line} and the exact same start time of {start_time}, are you sure this is correct?")
            

SyntaxError: invalid syntax. Perhaps you forgot a comma? (462758199.py, line 9)

In [ ]:
def 

In [ ]:
"""
Bus schedule file validator.

Design idea
-----------
Instead of raising on the first problem found (which forces the user to
fix-and-rerun one error at a time), this validator:
  1. Uses a small custom exception *hierarchy* so callers can catch
     specific problems (e.g. `except InvalidLocationError`) or the whole
     family (`except BusScheduleError`).
  2. Two checks are "fatal" and raise immediately, because nothing else
     can be validated without them: wrong file type, and missing columns.
  3. All *row-level* problems are collected into a list and raised together
     at the end as one `ScheduleValidationError`, so the user sees every
     problem in the file in one pass instead of one at a time.
"""

import os
import pandas as pd


# ---------------------------------------------------------------------------
# Custom exception hierarchy
# ---------------------------------------------------------------------------

class BusScheduleError(Exception):
    """Base class for every error this validator can raise."""

    def __init__(self, message, row=None):
        self.row = row
        if row is not None:
            message = f"[Row {row}] {message}"
        super().__init__(message)


class InvalidFileTypeError(BusScheduleError):
    """The uploaded file isn't a .csv, .xlsx, or .xls file."""


class MissingColumnsError(BusScheduleError):
    """One or more required columns are missing."""


class InvalidLocationError(BusScheduleError):
    """A start/end location isn't in the allowed list."""


class SameLocationError(BusScheduleError):
    """Start location and end location are identical for a row."""


class InvalidActivityError(BusScheduleError):
    """An activity value isn't one of the allowed activities."""


class EmptyActivityError(BusScheduleError):
    """The activity field is empty."""


class InvalidEnergyValueError(BusScheduleError):
    """Energy consumption isn't a valid number."""


class InvalidIdleEnergyError(BusScheduleError):
    """An idle activity doesn't consume exactly 5 energy."""


class InvalidLineNumberError(BusScheduleError):
    """Line isn't numeric, or has more than 3 digits."""


class DuplicateScheduleError(BusScheduleError):
    """Same bus + line + start time appears more than once."""


class ScheduleValidationError(BusScheduleError):
    """Aggregates every row-level error found in one validation pass."""

    def __init__(self, errors):
        self.errors = errors
        message = f"{len(errors)} validation error(s) found:\n" + "\n".join(
            f" - {e}" for e in errors
        )
        super().__init__(message)


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

REQUIRED_COLUMNS = [
    "start location", "end location", "start time", "end time",
    "activity", "line", "energy consumption", "bus",
]
ALLOWED_LOCATIONS = ["ehvbst", "ehvapt", "ehvgar"]
ALLOWED_ACTIVITIES = ["idle", "service trip", "material trip", "charging"]
ALLOWED_EXTENSIONS = {".csv", ".xlsx", ".xls"}


# ---------------------------------------------------------------------------
# Step 1: load + column check (fatal — raised immediately)
# ---------------------------------------------------------------------------

def load_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext not in ALLOWED_EXTENSIONS:
        raise InvalidFileTypeError(
            f"'{ext}' is not a supported file type. "
            f"Please provide a .csv, .xlsx, or .xls file."
        )
    if ext == ".csv":
        return pd.read_csv(path)
    return pd.read_excel(path)


def check_columns(df):
    missing = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing:
        raise MissingColumnsError(
            f"Missing columns: {sorted(missing)}. Please make sure all "
            f"required columns are present."
        )


# ---------------------------------------------------------------------------
# Step 2: row-level checks (non-fatal — collected into `errors`)
# ---------------------------------------------------------------------------

def check_locations(df, errors):
    bad_start = ~df["start location"].isin(ALLOWED_LOCATIONS)
    bad_end = ~df["end location"].isin(ALLOWED_LOCATIONS)
    for idx in df.index[bad_start | bad_end]:
        bad_val = df.loc[idx, "start location"] if bad_start[idx] else df.loc[idx, "end location"]
        errors.append(InvalidLocationError(f"Unexpected location '{bad_val}'.", row=idx))

    same = df["start location"] == df["end location"]
    for idx in df.index[same]:
        errors.append(SameLocationError("Start and end location are the same.", row=idx))


def check_activities(df, errors):
    empty = df["activity"].isnull()
    for idx in df.index[empty]:
        errors.append(EmptyActivityError("Activity is empty.", row=idx))

    invalid = ~df["activity"].isin(ALLOWED_ACTIVITIES) & ~empty
    for idx in df.index[invalid]:
        errors.append(InvalidActivityError(
            f"'{df.loc[idx, 'activity']}' is not one of {ALLOWED_ACTIVITIES}.", row=idx
        ))


def check_energy(df, errors):
    numeric_energy = pd.to_numeric(df["energy consumption"], errors="coerce")

    for idx in df.index[numeric_energy.isna()]:
        errors.append(InvalidEnergyValueError(
            f"'{df.loc[idx, 'energy consumption']}' is not a number.", row=idx
        ))

    idle_mask = (df["activity"] == "idle") & numeric_energy.notna()
    bad_idle = idle_mask & (numeric_energy != 5)
    for idx in df.index[bad_idle]:
        errors.append(InvalidIdleEnergyError(
            f"Idle activity must consume exactly 5 energy, got {numeric_energy[idx]}.",
            row=idx,
        ))


def check_lines(df, errors):
    for idx, value in df["line"].items():
        if pd.isna(value) or str(value).strip() == "":
            continue  # empty line is allowed

        numeric_value = pd.to_numeric(value, errors="coerce")
        if pd.isna(numeric_value):
            errors.append(InvalidLineNumberError(f"'{value}' is not a valid line number.", row=idx))
        elif len(str(int(numeric_value))) > 3:
            errors.append(InvalidLineNumberError(f"'{value}' has more than 3 digits.", row=idx))


def check_duplicates(df, errors):
    dupes = df[df.duplicated(subset=["bus", "line", "start time"], keep=False)]
    for (bus, line, start), group in dupes.groupby(["bus", "line", "start time"]):
        rows = list(group.index)
        errors.append(DuplicateScheduleError(
            f"Bus {bus}, line {line} has duplicate start time {start} in rows {rows}."
        ))


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def validate_file(path):
    df = load_file(path)   # fatal: can't validate anything without a dataframe
    check_columns(df)      # fatal: can't validate rows without required columns

    errors = []
    check_locations(df, errors)
    check_activities(df, errors)
    check_energy(df, errors)
    check_lines(df, errors)
    check_duplicates(df, errors)

    if errors:
        raise ScheduleValidationError(errors)

    return df


if __name__ == "__main__":
    import sys

    try:
        result = validate_file(sys.argv[1])
        print("File is valid.")
        print(result.head())
    except BusScheduleError as e:
        print(f"Validation failed: {e}")